## Model From Scratch

In [6]:
import torch
import torch.nn as nn

class MCQTransformer(nn.Module):

    def __init__(
        self,
        vocab_size,
        hidden_dim=256,
        max_length=256,
        num_heads=8,
        num_layers=4,
        dropout=0.1,
    ):
        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            hidden_dim
        )

        self.position_embedding = nn.Embedding(
            max_length,
            hidden_dim
        )

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden_dim,
            nhead=num_heads,
            dim_feedforward=hidden_dim * 4,
            dropout=dropout,
            batch_first=True,
            activation="gelu"
        )

        self.encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers
        )

        self.dropout = nn.Dropout(dropout)

        self.classifier = nn.Linear(hidden_dim, 1)

    def forward(self, input_ids):

        B, C, L = input_ids.shape

        input_ids = input_ids.view(B * C, L)

        positions = (
            torch.arange(L, device=input_ids.device)
            .unsqueeze(0)
            .expand(B * C, L)
        )

        x = self.embedding(input_ids)

        x = x + self.position_embedding(positions)

        x = self.encoder(x)

        x = x.mean(dim=1)

        x = self.dropout(x)

        logits = self.classifier(x)

        logits = logits.view(B, C)

        return logits

In [7]:
criterion = nn.CrossEntropyLoss()
loss = criterion(
    logits,
    labels
)

In [8]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=3e-4
)

for batch in train_loader:

    optimizer.zero_grad()
    logits = model(batch["input_ids"])
    loss = criterion(
        logits,
        batch["labels"]
    )

    loss.backward()
    optimizer.step()

In [9]:
test_dataset = MCQDataset(
    test,
    tokenizer,
    max_length=256
)

pred = trainer.predict(test_dataset)
logits = pred.predictions

In [10]:
import numpy as np

label_names = np.array(["A", "B", "C", "D", "E"])
top3 = np.argsort(-logits, axis=1)[:, :3]

predictions = [
    " ".join(label_names[idx])
    for idx in top3
]

submission = test[["id"]].copy()
submission["Prediction"] = predictions
submission.to_csv("submission.csv", index=False)

## Deberta-V3

In [27]:
import pandas as pd
import numpy as np
import torch
from sklearn.model_selection import GroupKFold
from transformers import AutoTokenizer, AutoModelForMultipleChoice, Trainer, TrainingArguments
from datasets import Dataset

MODEL_NAME = "microsoft/deberta-v3-base"
OPTIONS = ['A', 'B', 'C', 'D', 'E']
MAP_LABEL = {opt: idx for idx, opt in enumerate(OPTIONS)}
INV_MAP_LABEL = {idx: opt for idx, opt in enumerate(OPTIONS)}
MAX_LENGTH = 256

train_df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
test_df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def compute_metrics(eval_preds):
    logits, labels = eval_preds
   
    top_3_preds = np.argsort(-logits, axis=1)[:, :3]
    
   
    score = 0.0
    for i in range(len(labels)):
        correct_idx = labels[i]
        pred_list = top_3_preds[i]
        if correct_idx == pred_list[0]:
            score += 1.0
        elif correct_idx == pred_list[1]:
            score += 0.5
        elif correct_idx == pred_list[2]:
            score += 1.0 / 3.0
            
    return {"map@3": score / len(labels)}

# 2. Reshaping into 5-Choice Tensors
def preprocess_fn(examples, is_test=False):
    first_sentences = []
    second_sentences = []

    for i in range(len(examples['prompt'])):
        prompt = examples['prompt'][i]
        first_sentences.extend([prompt] * 5)
        for opt in OPTIONS:
            second_sentences.append(str(examples[opt][i]))
            

    tokenized = tokenizer(
        first_sentences,
        second_sentences,
        truncation="only_first", 
        max_length=MAX_LENGTH,
        padding="max_length"
    )
    
   
    features = {
        k: [v[i : i + 5] for i in range(0, len(v), 5)] 
        for k, v in tokenized.items()
    }
    
    if not is_test:
        features['label'] = [MAP_LABEL[ans] for ans in examples['answer']]
        
    return features


gkf = GroupKFold(n_splits=5)
train_df['fold'] = -1
for fold, (train_idx, val_idx) in enumerate(gkf.split(train_df, groups=train_df['prompt'])):
    train_df.loc[val_idx, 'fold'] = fold

train_split = train_df[train_df['fold'] != 0].reset_index(drop=True)
val_split = train_df[train_df['fold'] == 0].reset_index(drop=True)


train_ds = Dataset.from_pandas(train_split).map(preprocess_fn, batched=True, fn_kwargs={"is_test": False})
val_ds = Dataset.from_pandas(val_split).map(preprocess_fn, batched=True, fn_kwargs={"is_test": False})
test_ds = Dataset.from_pandas(test_df).map(preprocess_fn, batched=True, fn_kwargs={"is_test": True})


model = AutoModelForMultipleChoice.from_pretrained(MODEL_NAME)

for name, param in model.named_parameters():
    if param.requires_grad:
        param.data = param.data.float() 


training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=1.5e-5,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=4,
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    load_best_model_at_end=True,
    metric_for_best_model="map@3",
    greater_is_better=True,    
    logging_strategy="steps",
    logging_steps=5,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics
)

trainer.train()

predictions = trainer.predict(test_ds)
raw_logits = predictions.predictions 
top_3_indices = np.argsort(-raw_logits, axis=1)[:, :3]

submission_strings = []
for row in top_3_indices:
    submission_strings.append(" ".join([INV_MAP_LABEL[idx] for idx in row]))

submission = pd.DataFrame({
    'id': test_df['id'],
    'Prediction': submission_strings
})
submission.to_csv("submission.csv", index=False)


## DistilBert Model

In [38]:
!pip install -q transformers datasets accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 49.7 MB/s eta 0:00:0000:010:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12 26.2.1 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which 

In [39]:
import torch
import pandas as pd
import numpy as np

from transformers import (
    AutoTokenizer,
    AutoModelForMultipleChoice,
    Trainer,
    TrainingArguments
)

In [40]:
import pandas as pd

train = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
test = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")

print(train.shape)
train.head()

(2000, 8)


,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A


In [41]:
def build_example(prompt, option):
    return (
        f"Question:\n{prompt}\n\n"
        f"Candidate Answer:\n{option}"
    )

In [42]:
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()

train["label"] = encoder.fit_transform(train.answer)

In [43]:
MODEL_NAME = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForMultipleChoice.from_pretrained(
    MODEL_NAME
)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForMultipleChoice LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [44]:
from torch.utils.data import Dataset
import torch

class MCQDataset(Dataset):

    def __init__(self, df, tokenizer, max_length=256):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        question = row["prompt"]

        choices = [
            row["A"],
            row["B"],
            row["C"],
            row["D"],
            row["E"],
        ]

        encoding = self.tokenizer(
            [question] * 5,
            choices,
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt",
        )

        item = {
            "input_ids": encoding["input_ids"],
            "attention_mask": encoding["attention_mask"],
        }

        # Only add labels for train/validation
        if "label" in self.df.columns:
            item["labels"] = torch.tensor(row["label"], dtype=torch.long)

        return item

In [45]:
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()

train["label"] = encoder.fit_transform(train["answer"])

In [46]:
import numpy as np

def map3(eval_pred):

    logits, labels = eval_pred

    preds = np.argsort(-logits, axis=1)

    score = 0

    for p, y in zip(preds, labels):

        top3 = p[:3]

        if y == top3[0]:
            score += 1

        elif y == top3[1]:
            score += 0.5

        elif y == top3[2]:
            score += 1/3

    return {"map3": score / len(labels)}

In [47]:
from sklearn.model_selection import train_test_split

train_df, valid_df = train_test_split(
    train,
    test_size=0.2,
    stratify=train["label"],
    random_state=42
)

train_df = train_df.reset_index(drop=True)
valid_df = valid_df.reset_index(drop=True)

train_dataset = MCQDataset(
    train_df,
    tokenizer,
    max_length=256
)

valid_dataset = MCQDataset(
    valid_df,
    tokenizer,
    max_length=256
)

In [48]:

training_args = TrainingArguments(

    output_dir="./outputs",

    learning_rate=2e-5,

    per_device_train_batch_size=8,

    per_device_eval_batch_size=8,

    num_train_epochs=5,

    weight_decay=0.01,

    warmup_ratio=0.1,

    fp16=False,

    max_grad_norm=1.0,

    logging_steps=50,

    eval_strategy="epoch",

    save_strategy="epoch",

    load_best_model_at_end=True,

    metric_for_best_model="map3",

    greater_is_better=True,

    optim="adamw_torch"
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [49]:
trainer = Trainer(

    model=model,

    args=training_args,

    train_dataset=train_dataset,

    eval_dataset=valid_dataset,

    processing_class=tokenizer,

    compute_metrics=map3
)

trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Map3
1,2.949159,2.350422,0.820000
2,1.272614,1.254317,0.947500
3,0.739369,0.503742,0.981250
4,0.568357,0.379452,0.973750
5,0.399577,0.335746,0.977917


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=500, training_loss=1.3155822372436523, metrics={'train_runtime': 517.6894, 'train_samples_per_second': 15.453, 'train_steps_per_second': 0.966, 'total_flos': 2649300725760000.0, 'train_loss': 1.3155822372436523, 'epoch': 5.0})

In [50]:
test_dataset = MCQDataset(
    test,
    tokenizer,
    max_length=256
)

In [51]:
pred = trainer.predict(test_dataset)

logits = pred.predictions

labels = np.array(["A", "B", "C", "D", "E"])

order = np.argsort(-logits, axis=1)

predictions = [
    " ".join(labels[idx[:3]])
    for idx in order
]

In [52]:
print(len(pred.predictions))

500


In [53]:
submission = pd.DataFrame({
    "id": test["id"],
    "Prediction": predictions
})

submission.to_csv("submission.csv", index=False)

In [54]:
sample = train_dataset[0]

print(sample["input_ids"].shape)
print(sample["attention_mask"].shape)
print(sample["labels"])

outputs = model(
    input_ids=sample["input_ids"].unsqueeze(0).to(model.device),
    attention_mask=sample["attention_mask"].unsqueeze(0).to(model.device),
    labels=sample["labels"].unsqueeze(0).to(model.device),
)

print(outputs.loss)
print(outputs.logits)

torch.Size([5, 256])
torch.Size([5, 256])
tensor(2)
tensor(0.0127, device='cuda:0', grad_fn=<NllLossBackward0>)
tensor([[-3.0368, -2.2221,  3.5952, -1.3751, -2.8960]], device='cuda:0',
       grad_fn=<ViewBackward0>)


In [55]:
# Convert the scalar label to a PyTorch tensor first
label_tensor = torch.tensor(sample["labels"])

outputs = model(
    input_ids=sample["input_ids"].unsqueeze(0).to(model.device),
    attention_mask=sample["attention_mask"].unsqueeze(0).to(model.device),
    labels=label_tensor.unsqueeze(0).to(model.device), 
)

print(outputs.loss)
print(outputs.logits)


tensor(0.0127, device='cuda:0', grad_fn=<NllLossBackward0>)
tensor([[-3.0368, -2.2221,  3.5952, -1.3751, -2.8960]], device='cuda:0',
       grad_fn=<ViewBackward0>)


/tmp/ipykernel_58/1085475254.py:2: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  label_tensor = torch.tensor(sample["labels"])


In [56]:
import torch
batch = train_dataset[0]

with torch.no_grad():
    out = model(
        input_ids=batch["input_ids"].unsqueeze(0).to(model.device),
        attention_mask=batch["attention_mask"].unsqueeze(0).to(model.device),
    )

print(out.logits)

tensor([[-3.0368, -2.2221,  3.5952, -1.3751, -2.8960]], device='cuda:0')


## RAG

In [ ]:
!pip install pymupdf faiss-cpu wandb datasets -q

In [ ]:
import fitz   # pymupdf
import pandas as pd
import numpy as np
import re
import os

def extract_pdf_chunks(pdf_path, chunk_size=80, overlap=20):
    
    doc   = fitz.open(pdf_path)
    pages = []

    for page_num in range(len(doc)):
        page = doc[page_num]
        text = page.get_text()

        text = re.sub(r'\s+', ' ', text)          
        text = re.sub(r'[^\w\s\.\,\;\:\-\(\)]', ' ', text)  # remove special chars
        text = text.strip()

        if len(text) > 50:   # skip very short pages
            pages.append({
                'page': page_num + 1,
                'text': text
            })

    doc.close()

    full_text = ' '.join([p['text'] for p in pages])
    words = full_text.split()
    
    chunks = []
    i = 0
    while i < len(words):
        chunk_words = words[i:i + chunk_size]
        chunk_text  = ' '.join(chunk_words)

        if len(chunk_text) > 50:   # skip tiny chunks
            chunks.append(chunk_text)

        i += (chunk_size - overlap)   # overlap for context continuity

    return chunks


PDF_DIR = '/kaggle/input/datasets/aadity7531/rag-dataset'

pdfs = {
    'Physics_11_part1':   'NCERT-Class-11-Physics-Part-1.pdf',
    'Physics_11_part2':   'NCERT-Class-11-Physics-Part-2.pdf',
    'Physics_12_part1':   'NCERT-Class-12-Physics-Part-1.pdf',
    'Physics_12_part2':   'NCERT-Class-12-Physics-Part-2.pdf',
    'Chemistry_11_part1': 'NCERT-Class-11-Chemistry-Part-1.pdf',
    'Chemistry_11_part2': 'NCERT-Class-11-Chemistry-Part-2.pdf',
    'Chemistry_12_part1': 'NCERT-Class-12-Chemistry-Part-1.pdf',
    'Chemistry_12_part2': 'NCERT-Class-12-Chemistry-Part-2.pdf',
    'geography1': 'Fundamental of Physical Geography (Class XI) 2.pdf',
    'geography2': 'India Physical Environment (Class XI) 2.pdf',
    'geography3': '/kaggle/input/datasets/aadity7531/rag-dataset/Practical Work in Geography Part 1.pdf',
    'class9': '/kaggle/input/datasets/aadity7531/rag-dataset/class 9 abc.pdf',
    'history1': '/kaggle/input/datasets/aadity7531/rag-dataset/NCERT-Class-10-History.pdf',
    'history2': '/kaggle/input/datasets/aadity7531/rag-dataset/NCERT-Class-11-History.pdf',
    'Biology_11': '/kaggle/input/datasets/aadity7531/rag-dataset/NCERT-Class-11-Biology.pdf',
    'Biology_12': '/kaggle/input/datasets/aadity7531/rag-dataset/NCERT-Class-12-Biology.pdf',
    'history_12_1': '/kaggle/input/datasets/aadity7531/rag-dataset/NCERT-Class-12-History-Part-1 (1).pdf',
    'History_12_2': '/kaggle/input/datasets/aadity7531/rag-dataset/NCERT-Class-12-History-Part-2.pdf',
    'History_12_3': '/kaggle/input/datasets/aadity7531/rag-dataset/NCERT-Class-12-History-Part-3.pdf',
    
}

all_chunks = []

for subject, filename in pdfs.items():
    path = os.path.join(PDF_DIR, filename)
    
    chunks = extract_pdf_chunks(path, chunk_size=100, overlap=30)

    for chunk in chunks:
        all_chunks.append({
            'subject': subject,
            'text':    chunk
        })

    print(f"{subject}: {len(chunks)} chunks extracted")

# Save to dataframe
knowledge_df = pd.DataFrame(all_chunks)
print(knowledge_df.iloc[10]['text'][:300])

In [ ]:
import faiss
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize

docs = knowledge_df['text'].tolist()

# TF-IDF vectorizer on NCERT text
tfidf = TfidfVectorizer(
    max_features=15000,
    ngram_range=(1, 2),
    min_df=2,
    strip_accents='unicode'
)

doc_vecs = tfidf.fit_transform(docs).toarray().astype(np.float32)
doc_vecs  = normalize(doc_vecs, norm='l2')

# FAISS index
DIM   = doc_vecs.shape[1]
index = faiss.IndexFlatIP(DIM)
index.add(doc_vecs)

print(f"FAISS ready — {index.ntotal} knowledge chunks, dim={DIM}")

In [ ]:
def get_ncert_context(prompt: str, options: dict, top_k: int = 3) -> str:

    query = prompt + ' ' + ' '.join(options.values())
    query = query[:500]   # limit query length

    vec = tfidf.transform([query]).toarray().astype(np.float32)
    vec = normalize(vec, norm='l2')

    scores, idxs = index.search(vec, top_k + 2)

    parts = []
    for score, idx in zip(scores[0], idxs[0]):
        if idx < 0:
            continue
        if score < 0.15:       # skip irrelevant chunks
            continue
        if len(parts) >= top_k:
            break

        chunk = docs[int(idx)][:250]   # truncate for token budget
        parts.append(chunk)

    return ' || '.join(parts) if parts else 'No relevant context.'



test_q = "What is the relationship between Hamiltonians in quantum mechanics?"
test_opts = {'A': 'same energy', 'B': 'higher energy', 'C': 'different spin', 'D': 'different energy', 'E': 'lower energy'}

ctx = get_ncert_context(test_q, test_opts)
print("Query:", test_q)
print("\nRetrieved NCERT context:")
print(ctx[:400])

In [ ]:
import pandas as pd

train = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
test  = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv')

OPTIONS  = ['A', 'B', 'C', 'D', 'E']
LABEL2ID = {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4}
ID2LABEL = {0: 'A', 1: 'B', 2: 'C', 3: 'D', 4: 'E'}

for col in OPTIONS + ['prompt']:
    train[col] = train[col].fillna('none')
    test[col]  = test[col].fillna('none')

train['label'] = train['answer'].map(LABEL2ID)
train = train.reset_index(drop=True)
test  = test.reset_index(drop=True)

# Build contexts
print("Building NCERT RAG context for train...")
train_ctx = []
for i in range(len(train)):
    opts = {opt: train.iloc[i][opt] for opt in OPTIONS}
    ctx  = get_ncert_context(train.iloc[i]['prompt'], opts)
    train_ctx.append(ctx)
    if i % 400 == 0:
        print(f"  {i}/2000")

train['rag_context'] = train_ctx

test_ctx = []
for i in range(len(test)):
    opts = {opt: test.iloc[i][opt] for opt in OPTIONS}
    ctx  = get_ncert_context(test.iloc[i]['prompt'], opts)
    test_ctx.append(ctx)
    if i % 100 == 0:
        print(f"  {i}/500")

test['rag_context'] = test_ctx

real_ctx = (train['rag_context'] != 'No relevant context.').sum()
print(f"Questions with real NCERT context: {real_ctx}/2000 ({real_ctx/20:.1f}%)")

In [ ]:
import torch
from transformers import AutoTokenizer
from datasets import Dataset

DEVICE     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
MODEL_NAME = '/kaggle/input/models/mtnash/deberta-v3-small/transformers/default/1'
MAX_LEN    = 256
EPOCHS     = 2
LR         = 1e-5
BATCH_SIZE = 4

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_fn(examples):
    all_input_ids      = []
    all_attention_mask = []

    for i in range(len(examples['prompt'])):
        
        ctx    = str(examples['rag_context'][i])[:150]
        prompt = str(examples['prompt'][i])

        # Format: "Context: <ncert_text> Question: <prompt>"
        first_seq = f"Context: {ctx} Question: {prompt}"

        opt_ids = []
        opt_att = []

        for opt in OPTIONS:
            second_seq = str(examples[opt][i])

            enc = tokenizer(
                first_seq,
                second_seq,
                max_length=MAX_LEN,
                truncation=True,
                padding='max_length',
                return_tensors=None,
            )
            opt_ids.append([int(x) for x in enc['input_ids']])
            opt_att.append([int(x) for x in enc['attention_mask']])

        all_input_ids.append(opt_ids)
        all_attention_mask.append(opt_att)

    return {
        'input_ids':      all_input_ids,
        'attention_mask': all_attention_mask,
    }

In [ ]:
from sklearn.model_selection import train_test_split

KEEP     = OPTIONS + ['prompt', 'rag_context']
KEEP_LBL = KEEP + ['label']

# Use full 2000 for training
train_hf = Dataset.from_pandas(train[KEEP_LBL].copy())
test_hf  = Dataset.from_pandas(test[KEEP].copy())


train_tok = train_hf.map(tokenize_fn, batched=True, batch_size=32, remove_columns=KEEP)

test_tok  = test_hf.map(tokenize_fn, batched=True, batch_size=32, remove_columns=KEEP)

FEAT = ['input_ids', 'attention_mask']
train_tok.set_format(type='torch', columns=FEAT + ['label'])
test_tok.set_format( type='torch', columns=FEAT)

print("Done!")
print("Shape:", train_tok[0]['input_ids'].shape)  # (5, 256)

In [ ]:
import gc
import wandb
import torch
import torch.nn as nn
import pandas as pd
from torch.optim import AdamW
from torch.utils.data import DataLoader
from transformers import AutoModelForMultipleChoice, get_cosine_schedule_with_warmup
from sklearn.metrics import f1_score, accuracy_score

def collate_train(features):
    return {
        'input_ids':      torch.stack([f['input_ids']      for f in features]).long(),
        'attention_mask': torch.stack([f['attention_mask'] for f in features]).long(),
        'labels':         torch.stack([f['label']          for f in features]).long(),
    }

def collate_test(features):
    return {
        'input_ids':      torch.stack([f['input_ids']      for f in features]).long(),
        'attention_mask': torch.stack([f['attention_mask'] for f in features]).long(),
    }


train_loader = DataLoader(
    train_tok, batch_size=BATCH_SIZE,
    shuffle=True,  collate_fn=collate_train
)
test_loader = DataLoader(
    test_tok, batch_size=BATCH_SIZE,
    shuffle=False, collate_fn=collate_test,
)


wandb.login(key="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")
wandb.init(
    project='24f1002052-t22026',
    name='deberta-NCERT-RAG',
    config={
        'model':     MODEL_NAME,
        'knowledge': 'NCERT Physics+Chemistry 11+12',
        'epochs':    EPOCHS,
        'lr':        LR,
        'max_len':   MAX_LEN
    }
)


criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

model     = AutoModelForMultipleChoice.from_pretrained(
    MODEL_NAME, ignore_mismatched_sizes=True
).to(DEVICE)

optimizer = AdamW(model.parameters(), lr=LR, weight_decay=0.01)

total_steps = len(train_loader) * EPOCHS

scheduler   = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=total_steps // 10,
    num_training_steps=total_steps,
)

best_loss  = float('inf')
best_state = None

for epoch in range(EPOCHS):
    model.train()
    total_loss  = 0.0
    valid_steps = 0

    for step, batch in enumerate(train_loader):
        batch  = {k: v.to(DEVICE) for k, v in batch.items()}
        labels = batch.pop('labels')
        optimizer.zero_grad()
        
        out  = model(**batch)
        loss = criterion(out.logits, labels)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 0.5)
        optimizer.step()
        scheduler.step()

        total_loss  += loss.item()
        valid_steps += 1

        if step % 50 == 0:
            print(f"  Step {step}/{len(train_loader)} loss={loss.item():.4f}")

    avg = total_loss / max(valid_steps, 1)
    wandb.log({'epoch': epoch+1, 'train_loss': avg})

    if avg < best_loss:
        best_loss  = avg
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        print(f"  ✅ Best saved! Loss={avg:.4f}")

wandb.finish()
print("Training done!")


model.load_state_dict(best_state)
model.to(DEVICE)
model.eval()

all_preds = []
with torch.no_grad():
    for batch in test_loader:
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        out   = model(**batch)
        preds = torch.argmax(out.logits, dim=1)
        all_preds.extend(preds.cpu().numpy())

pred_labels = [ID2LABEL[int(p)] for p in all_preds]

sub = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv')
sub['Prediction'] = pred_labels
sub.to_csv('/kaggle/working/submission.csv', index=False)
print("✅ Submission saved!")
print(sub.head())